# Reliable Error Estimation for PINNs: Lower and Upper A Posteriori Bounds

**Paper:** Huseynov, I., Ahmadova, A., Bashirov, A. (2026). *Reliable Error Estimation for PINNs: Lower and Upper A Posteriori Bounds.* arXiv:2606.12050 [cs.LG].

**Carpeta origen:** `PINNs/4. Otros/Reliable Error Estimation for PINNs Lower and Upper A Posteriori Bounds.pdf`

## Como se usan las PINNs en este paper

Este paper no propone una nueva forma de entrenar PINNs, sino una forma de **certificar rigurosamente el error** de una PINN ya entrenada para una EDO $\dot x(t)=f(t,x(t))$, $x(0)=x_0$, **sin conocer la solucion exacta**. Definen el residuo de la trayectoria de la red $\hat x(t)$ (Eq. anterior al Teorema 1):

$$\mathcal{R}_{\hat\varphi}(t):=\dot{\hat x}(t)-f(t,\hat x(t))$$

calculable por diferenciacion automatica, y una cota computable $\delta(t)\geq\|\mathcal{R}_{\hat\varphi}(t)\|$. Bajo una condicion de monotonicidad local de un solo lado (Eq. 7), el **Teorema 1** da una **cota inferior** rigurosa para el error $e(t)=x(t)-\hat x(t)$:

$$\|e(t)\|\geq e^{\ell_Dt}\|e(0)\|-\int_0^t e^{\ell_D(t-s)}\delta(s)\,ds$$

y de forma complementaria (bajo la condicion de Lipschitz de un solo lado, Eq. 3), una **cota superior** analoga con $\mu_D$:

$$\|e(t)\|\leq e^{\mu_Dt}\|e(0)\|+\int_0^t e^{\mu_D(t-s)}\delta(s)\,ds$$

Para sistemas **lineales invariantes en el tiempo** $\dot x=Ax+g(t)$, $\ell_D$ y $\mu_D$ se calculan explicitamente como los autovalores minimo y maximo de la parte simetrica de $A$ (para el caso escalar, $\ell_D=\mu_D=\partial_xf$). Estas dos cotas juntas dan un **encierro certificado de dos lados** del error de la PINN &mdash; sin necesitar la solucion exacta, solo la red entrenada, su residuo, y las constantes de monotonicidad/Lipschitz locales del sistema.

Este cuaderno reproduce fielmente el mecanismo (Teorema 1 y su analogo superior) sobre una **EDO lineal escalar forzada** ($\ell_D=\mu_D$ exactos, calculables en forma cerrada), entrenando una PINN estandar y verificando que las cotas calculadas efectivamente **encierran el error real** (comparado contra la solucion exacta, usada aqui solo para validar visualmente que el encierro certificado es correcto, no para calcular las cotas en si).

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio en las paginas revisadas, ni se encontro uno especifico. Como referencia general del framework PINN base:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. EDO lineal escalar forzada: $\dot x=ax+b\sin(t)$, $x(0)=x_0$ ($\ell_D=\mu_D=a$, exacto para sistemas LTI escalares)

In [ ]:
a_coef, b_coef, x0_val, T_max = -2.0, 3.0, 1.0, 5.0

def f_rhs(t, x):
    return a_coef * x + b_coef * torch.sin(t)

def exact_x(t):
    """Solucion cerrada exacta (homogenea + particular) del sistema LTI:
    x_p(t) = -b/(1+a^2) * (a*sin(t) + cos(t)), x(t) = C*exp(a*t) + x_p(t)."""
    C = x0_val + b_coef / (a_coef**2 + 1)
    return C * np.exp(a_coef * t) - (b_coef / (a_coef**2 + 1)) * (a_coef * np.sin(t) + np.cos(t))

ell_D = mu_D = a_coef  # exacto para sistema lineal invariante en el tiempo escalar (Remark 1)
print(f'ell_D = mu_D = {ell_D} (calculado en forma cerrada, sin necesitar la solucion exacta)')

## 2. Entrenamiento de una PINN estandar (restriccion blanda de la condicion inicial)

In [ ]:
class ScalarPINN(nn.Module):
    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)


model = ScalarPINN().to(device)


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]


t_col = torch.linspace(0, T_max, 300, device=device).view(-1, 1).requires_grad_(True)
t0 = torch.zeros(1, 1, device=device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(4000):
    optimizer.zero_grad()
    x_hat = model(t_col)
    dx_hat = d_dt(x_hat, t_col)
    loss_ode = torch.mean((dx_hat - f_rhs(t_col, x_hat))**2)
    loss_ic = (model(t0) - x0_val)**2
    loss = loss_ode + 20.0 * loss_ic.squeeze()
    loss.backward()
    optimizer.step()
    if epoch % 1000 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e}')

## 3. Certificados a posteriori (Teorema 1 y su analogo superior): residuo, $\delta(t)$, e integrales de Gronwall

In [ ]:
t_eval = torch.linspace(0, T_max, 500, device=device).view(-1, 1).requires_grad_(True)
x_hat_eval = model(t_eval)
dx_hat_eval = d_dt(x_hat_eval, t_eval)
residual = (dx_hat_eval - f_rhs(t_eval, x_hat_eval)).detach().cpu().numpy().flatten()
t_np = t_eval.detach().cpu().numpy().flatten()

delta_t = np.abs(residual)  # cota computable del residuo (aqui, el propio |residuo|; Eq. 8)

e0 = abs(x0_val - model(t0).item())  # error en t=0 (aprox. blanda de la IC, no exactamente 0)

def gronwall_bounds(t_np, delta_t, e0, coef):
    lower = np.zeros_like(t_np)
    upper = np.zeros_like(t_np)
    for i, t in enumerate(t_np):
        mask = t_np <= t
        s = t_np[mask]
        integrand = np.exp(coef * (t - s)) * delta_t[mask]
        integral = np.trapezoid(integrand, s) if len(s) > 1 else 0.0
        lower[i] = max(np.exp(coef * t) * e0 - integral, 0.0)   # Eq. (9), Teorema 1
        upper[i] = np.exp(coef * t) * e0 + integral               # analogo superior
    return lower, upper

lower_bound, upper_bound = gronwall_bounds(t_np, delta_t, e0, ell_D)

## 4. Resultados: el error real debe quedar encerrado entre la cota inferior y la superior

In [ ]:
x_exact = exact_x(t_np)
with torch.no_grad():
    x_pred = model(t_eval).cpu().numpy().flatten()
true_error = np.abs(x_exact - x_pred)

plt.figure(figsize=(8, 5))
plt.plot(t_np, true_error, 'k-', linewidth=2, label='Error real |x(t) - x_hat(t)| (solo para verificar)')
plt.plot(t_np, lower_bound, 'b--', label='Cota inferior certificada (Teorema 1)')
plt.plot(t_np, upper_bound, 'r--', label='Cota superior certificada')
plt.fill_between(t_np, lower_bound, upper_bound, color='gray', alpha=0.2, label='Encierro certificado')
plt.xlabel('t'); plt.ylabel('Error')
plt.title('Certificado a posteriori de dos lados para el error de la PINN')
plt.legend()
plt.show()

contained = np.all((true_error >= lower_bound - 1e-6) & (true_error <= upper_bound + 1e-6))
print(f'El error real queda contenido en el encierro certificado en todo el intervalo: {contained}')